In [ ]:

# CONFIG
VIDEO_FOLDER = "videos"
OUTPUT_FILE = "results.xlsx"


In [ ]:

# RED DETECTION

def detect_red(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower_red_1 = np.array([0, 120, 70])
    upper_red_1 = np.array([10, 255, 255])

    lower_red_2 = np.array([170, 120, 70])
    upper_red_2 = np.array([180, 255, 255])

    mask1 = cv2.inRange(hsv, lower_red_1, upper_red_1)
    mask2 = cv2.inRange(hsv, lower_red_2, upper_red_2)

    mask = mask1 + mask2

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return contours

In [ ]:
# ANALYSIS FUNCTION

def analyze_video(video_path):
    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_index = 0

    start_frame = None
    end_frame = None

    time_series = []
    area_series = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        contours = detect_red(frame)

        if contours:
            largest = max(contours, key=cv2.contourArea)
            area = cv2.contourArea(largest)

            if start_frame is None:
                start_frame = frame_index

            end_frame = frame_index
        else:
            area = 0

        time_series.append(frame_index / fps)
        area_series.append(area)

        frame_index += 1

    cap.release()

    if start_frame is None or end_frame is None:
        return None, None

    duration = (end_frame - start_frame) / fps
    max_area = max(area_series)

    half_area = max_area * 0.5
    time_to_half = None

    for t, a in zip(time_series, area_series):
        if a <= half_area:
            time_to_half = t
            break

    summary = {
        "duration_sec": duration,
        "max_area": max_area,
        "time_to_half": time_to_half
    }

    curve = pd.DataFrame({
        "time": time_series,
        "area": area_series
    })

    return summary, curve


In [ ]:
# MAIN PIPELINE

def run():
    summary_rows = []
    all_curves = []

    for file in os.listdir(VIDEO_FOLDER):
        if file.lower().endswith(".mp4"):
            video_path = os.path.join(VIDEO_FOLDER, file)

            print(f"Processing {file}...")

            summary, curve_df = analyze_video(video_path)

            if summary is None:
                print(f"Skipping {file} (no detection)")
                continue

            # Add video_id to summary
            summary["video_id"] = file
            summary_rows.append(summary)

            # Add video_id to curve data
            curve_df["video_id"] = file
            all_curves.append(curve_df)

    # Combine everything
    df_summary = pd.DataFrame(summary_rows)

    if all_curves:
        df_curves = pd.concat(all_curves, ignore_index=True)
    else:
        df_curves = pd.DataFrame()

    # Save to Excel
    with pd.ExcelWriter(OUTPUT_FILE) as writer:
        df_summary.to_excel(writer, sheet_name="summary", index=False)
        df_curves.to_excel(writer, sheet_name="curves", index=False)

    print("Done. Results saved to", OUTPUT_FILE)

In [ ]:
# ENTRY POINT

if __name__ == "__main__":
    run()